# Drill 3: causal attention, KV cache, attention backward (25 min + bonus)

**Part A.** Multi-head causal self-attention with a KV cache. The trap: when you call it with a cache *and* several
new tokens (chunked prefill), the causal mask must use absolute positions. `is_causal=True` in PyTorch aligns the
mask to the top-left corner, which is wrong there.

**Part B.** The attention backward pass by hand: `dV = P^T dO`, `dP = dO V^T`, `dS = P * (dP - rowsum(dP * P))`,
`dQ = dS K / sqrt(d)`, `dK = dS^T Q / sqrt(d)`. Try to derive it before you look at this line again.

**Part C (bonus).** Online softmax: attention over key blocks without the full score matrix.

In [ ]:
import inspect
import torch
import torch.nn.functional as F

## Part A

In [ ]:
# Example inputs: a scratch cell to inspect shapes and try operations. The tests define their own inputs.
B, T, D, H = 2, 5, 8, 2
x = torch.randn(B, T, D)                                             # (B, T, D)
Wq, Wk, Wv, Wo = (torch.randn(D, D) for _ in range(4))               # each (D, D)
x_new = torch.randn(B, 1, D)                                         # (B, 1, D) one more token, for the cache

In [ ]:
def mha(
    x,               # (B, T, D)       T new tokens
    Wq, Wk, Wv, Wo,  # each (D, D)     projections are x @ W
    n_heads,         # int H           head dim dh = D // H
    cache=None,      # None, or {"k": (B, H, T_past, dh), "v": (B, H, T_past, dh)} from earlier calls
):
    """Causal multi-head self-attention with a KV cache, written by hand (no built-in attention function).
    The T new tokens sit at absolute positions T_past .. T_past + T - 1; each attends to keys at or before its position.

    Returns:
        out        (B, T, D)
        new_cache  {"k": (B, H, T_past + T, dh), "v": (B, H, T_past + T, dh)}
    """
    # TODO
    raise NotImplementedError


In [ ]:
torch.manual_seed(0)
B, T, D, H = 2, 9, 16, 4
x = torch.randn(B, T, D, dtype=torch.float64)                        # (B, T, D)
Ws = [torch.randn(D, D, dtype=torch.float64) / D ** 0.5 for _ in range(4)]


def reference(x, Wq, Wk, Wv, Wo, H):
    B, T, D = x.shape
    q, k, v = [(x @ W).view(B, T, H, D // H).transpose(1, 2) for W in (Wq, Wk, Wv)]
    o = F.scaled_dot_product_attention(q, k, v, is_causal=True)
    return o.transpose(1, 2).reshape(B, T, D) @ Wo


assert "scaled_dot_product" not in inspect.getsource(mha), "write attention by hand"
full = reference(x, *Ws, H)

In [ ]:
out, cache = mha(x, *Ws, H)
assert out.shape == (B, T, D)
assert torch.allclose(out, full, atol=1e-10), "full forward does not match"
assert cache["k"].shape == (B, H, T, D // H)
print("ok")

In [ ]:
# token-by-token decoding with the cache == full forward
cache, outs = None, []
for t in range(T):
    o, cache = mha(x[:, t:t + 1], *Ws, H, cache=cache)
    outs.append(o)
assert torch.allclose(torch.cat(outs, 1), full, atol=1e-10), "decoding with the cache does not match"
print("ok")

In [ ]:
# prefill 5 tokens, then a chunk of 4 new tokens with the cache (absolute positions matter here)
o1, cache = mha(x[:, :5], *Ws, H)
o2, cache = mha(x[:, 5:], *Ws, H, cache=cache)
assert torch.allclose(o1, full[:, :5], atol=1e-10)
assert torch.allclose(o2, full[:, 5:], atol=1e-10), "chunk after prefill is wrong: check the mask offset"
print("ok")

In [ ]:
# causality: changing token 6 changes nothing before it
x2 = x.clone()
x2[:, 6] += 1.0
o_a, _ = mha(x, *Ws, H)
o_b, _ = mha(x2, *Ws, H)
assert torch.allclose(o_a[:, :6], o_b[:, :6], atol=1e-12), "future token leaked into the past"
assert not torch.allclose(o_a[:, 6:], o_b[:, 6:])
print("ok")

## Part B

In [ ]:
# Example inputs: a scratch cell to inspect shapes and try operations. The tests define their own inputs.
T, d = 4, 3
q, k, v = torch.randn(T, d), torch.randn(T, d), torch.randn(T, d)    # each (T, d)
dout = torch.randn(T, d)                                             # (T, d) gradient arriving at the output

In [ ]:
def attn_forward_backward(
    q,     # (T, d)
    k,     # (T, d)
    v,     # (T, d)
    dout,  # (T, d)   gradient of the loss w.r.t. the output
):
    """One head, causal, scale 1/sqrt(d). Computed by hand: the test runs under torch.no_grad.

    Returns:
        out  (T, d)
        dq   (T, d)
        dk   (T, d)
        dv   (T, d)
    """
    # TODO
    raise NotImplementedError


In [ ]:
torch.manual_seed(1)
T_, d_ = 7, 5
q, k, v, dout = [torch.randn(T_, d_, dtype=torch.float64) for _ in range(4)]
with torch.no_grad():
    out, dq, dk, dv = attn_forward_backward(q, k, v, dout)

qa, ka, va = [t.clone().requires_grad_() for t in (q, k, v)]
oa = F.scaled_dot_product_attention(qa[None], ka[None], va[None], is_causal=True)[0]
oa.backward(dout)
assert torch.allclose(out, oa, atol=1e-10), "forward is wrong"
for name, mine, ref in [("dq", dq, qa.grad), ("dk", dk, ka.grad), ("dv", dv, va.grad)]:
    assert torch.allclose(mine, ref, atol=1e-10), f"{name} is wrong"
print("ok")

## Part C (bonus)

In [ ]:
# Example inputs: a scratch cell to inspect shapes and try operations. The tests define their own inputs.
q = torch.randn(3, 4)                                                # (Tq, d)
k, v = torch.randn(10, 4), torch.randn(10, 4)                        # (Tk, d)
block = 4

In [ ]:
def attention_online(
    q,         # (Tq, d)
    k,         # (Tk, d)
    v,         # (Tk, d)
    block=4,   # int   rows of k, v per block
):
    """Bonus. One head, no mask. Visit k, v in blocks, keeping per query a running max (Tq, 1), a running softmax
    denominator (Tq, 1) and a running output (Tq, d) (FlashAttention's trick). Never build the (Tq, Tk) matrix.

    Returns:
        out  (Tq, d)
    """
    # TODO
    raise NotImplementedError


In [ ]:
torch.manual_seed(2)
q, k, v = torch.randn(6, 8, dtype=torch.float64) * 3, torch.randn(11, 8, dtype=torch.float64) * 3, torch.randn(11, 8, dtype=torch.float64)
ref = torch.softmax(q @ k.T / 8 ** 0.5, -1) @ v
for block in [1, 3, 4, 11]:
    assert torch.allclose(attention_online(q, k, v, block), ref, atol=1e-10), f"block={block}"
print("ok")